In [86]:
import warnings
from glob import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from category_encoders import OneHotEncoder
from IPython.display import VimeoVideo
from sklearn.linear_model import LinearRegression, Ridge  # noqa F401
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.utils.validation import check_is_fitted

warnings.simplefilter(action="ignore", category=FutureWarning)

In [87]:
def wrangle(filepath):
    # Read CSV file
    df = pd.read_csv(filepath)

    # Subset data: Apartments in "Capital Federal", less than 400,000
    mask_ba = df["place_with_parent_names"].str.contains("Capital Federal")
    mask_apt = df["property_type"] == "apartment"
    mask_price = df["price_aprox_usd"] < 400_000
    df = df[mask_ba & mask_apt & mask_price]

    # Subset data: Remove outliers for "surface_covered_in_m2"
    low, high = df["surface_covered_in_m2"].quantile([0.1, 0.9])
    mask_area = df["surface_covered_in_m2"].between(low, high)
    df = df[mask_area]

    # Split "lat-lon" column
    df[["lat", "lon"]] = df["lat-lon"].str.split(",", expand=True).astype(float)
    df.drop(columns="lat-lon", inplace=True)
    
    #Extract Neighbourhood
    df["neighbourhood"] = df["place_with_parent_names"].str.split("|",expand=True)[3]
    df.drop(columns=["place_with_parent_names"],inplace=True);

    

    return df

In [88]:
files = glob("data/buenos-aires-real-estate-*.csv")
files

['data/buenos-aires-real-estate-3.csv',
 'data/buenos-aires-real-estate-2.csv',
 'data/buenos-aires-real-estate-1.csv',
 'data/buenos-aires-real-estate-5.csv',
 'data/buenos-aires-real-estate-4.csv']

In [89]:
assert len(files) == 5, f"`files` should contain 5 items, not {len(files)}"

In [90]:
frames = []
for file in files: 
    df = wrangle(file);
    frames.append(df);


In [91]:
type(frames[0])

pandas.core.frame.DataFrame

In [92]:
# Check your work
assert len(frames) == 5, f"`frames` should contain 5 items, not {len(frames)}"
assert all(
    [isinstance(frame, pd.DataFrame) for frame in frames]
), "The items in `frames` should all be DataFrames."

In [93]:
df = pd.concat(frames,ignore_index=True)
df.head()

,operation,property_type,price,currency,price_aprox_local_currency,price_aprox_usd,surface_total_in_m2,surface_covered_in_m2,price_usd_per_m2,price_per_m2,floor,rooms,expenses,properati_url,lat,lon,neighbourhood
0,sell,apartment,120000.0,USD,1819488.00,120000.0,NaN,55.0,NaN,2181.818182,NaN,2.0,NaN,http://villa-general-mitre.properati.com.ar/xx...,-34.616004,-58.470506,Villa General Mitre
1,sell,apartment,89000.0,USD,1349453.60,89000.0,NaN,37.0,NaN,2405.405405,7.0,2.0,NaN,http://palermo.properati.com.ar/ya5i_venta_dep...,-34.584712,-58.444927,Palermo
2,sell,apartment,183495.0,USD,2782224.58,183495.0,92.0,57.0,1994.51087,3219.210526,NaN,2.0,NaN,http://saavedra.properati.com.ar/12izq_venta_d...,-34.554652,-58.493644,Saavedra
3,sell,apartment,95000.0,USD,1440428.00,95000.0,53.0,47.0,1792.45283,2021.276596,NaN,2.0,NaN,http://villa-del-parque.properati.com.ar/wy0n_...,-34.610581,-58.479625,Villa del Parque
4,sell,apartment,95000.0,USD,1440428.00,95000.0,0.0,35.0,NaN,2714.285714,NaN,1.0,NaN,http://belgrano.properati.com.ar/xw9a_venta_de...,-34.558227,-58.458357,Belgrano


In [94]:
df.shape

(6582, 17)

In [95]:
# Check your work
assert len(df) == 6582, f"`df` is the wrong size: {len(df)}."

In [96]:
#Test Cell on how to extract neighbourhood and delete the place with parent names
#df["neighbourhood"] = df["place_with_parent_names"].str.split("|",expand=True)[3]
#df.drop(columns=["place_with_parent_names"],inplace=True);

In [97]:
frames[0]

,operation,property_type,price,currency,price_aprox_local_currency,price_aprox_usd,surface_total_in_m2,surface_covered_in_m2,price_usd_per_m2,price_per_m2,floor,rooms,expenses,properati_url,lat,lon,neighbourhood
7,sell,apartment,120000.0,USD,1819488.00,120000.0,NaN,55.0,NaN,2181.818182,NaN,2.0,NaN,http://villa-general-mitre.properati.com.ar/xx...,-34.616004,-58.470506,Villa General Mitre
20,sell,apartment,89000.0,USD,1349453.60,89000.0,NaN,37.0,NaN,2405.405405,7.0,2.0,NaN,http://palermo.properati.com.ar/ya5i_venta_dep...,-34.584712,-58.444927,Palermo
21,sell,apartment,183495.0,USD,2782224.58,183495.0,92.0,57.0,1994.510870,3219.210526,NaN,2.0,NaN,http://saavedra.properati.com.ar/12izq_venta_d...,-34.554652,-58.493644,Saavedra
41,sell,apartment,95000.0,USD,1440428.00,95000.0,53.0,47.0,1792.452830,2021.276596,NaN,2.0,NaN,http://villa-del-parque.properati.com.ar/wy0n_...,-34.610581,-58.479625,Villa del Parque
43,sell,apartment,95000.0,USD,1440428.00,95000.0,0.0,35.0,NaN,2714.285714,NaN,1.0,NaN,http://belgrano.properati.com.ar/xw9a_venta_de...,-34.558227,-58.458357,Belgrano
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8574,sell,apartment,145000.0,USD,2198548.00,145000.0,50.0,50.0,2900.000000,2900.000000,NaN,NaN,NaN,http://belgrano.properati.com.ar/e4x0_venta_de...,-34.553534,-58.436914,Belgrano
8577,sell,apartment,149000.0,USD,2259197.60,149000.0,53.0,50.0,2811.320755,2980.000000,NaN,NaN,NaN,http://recoleta.properati.com.ar/11a35_venta_d...,-34.583538,-58.405318,Recoleta
8582,sell,apartment,118000.0,USD,1789163.20,118000.0,40.0,40.0,2950.000000,2950.000000,NaN,2.0,1790.0,http://belgrano.properati.com.ar/zugq_venta_de...,-34.568917,-58.457627,Belgrano
8593,sell,apartment,77800.0,USD,1179634.72,77800.0,NaN,36.0,NaN,2161.111111,NaN,2.0,NaN,http://san-cristobal.properati.com.ar/1164k_ve...,-34.622790,-58.394566,San Cristobal
